# Error Analysis

Looking at real misclassified examples from the best model (Model 3: fine-tuned Sentence Transformer) and, separately, from Model 4 (LLM few-shot) -- not EDA-style heuristics on raw features, but actual model predictions vs. true labels.

Uses `find_mismatched_cases` from `src/utils.py` (already written for this purpose) to pull:
- **False positives**: score above threshold, but not actually a duplicate -- pairs that *look* similar but aren't
- **False negatives**: score below threshold, but actually a duplicate -- pairs that *are* the same question but don't score as similar (e.g. synonyms, very different phrasing)

Model 3's saved weights (`models/sentence_transformer_duplicate_model/`) are used to run inference locally (CPU) on a sample of val -- not the full ~64,686 rows, since CPU encoding is much slower than the Colab GPU used for training. A few thousand rows is enough to surface real hard cases for qualitative analysis; it doesn't need to be exhaustive.

## Setup

In [74]:
import sys
sys.path.append('..')

In [75]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import paired_cosine_distances
from sentence_transformers import SentenceTransformer

from src.utils import find_mismatched_cases

pd.set_option('display.max_colwidth', None)

### Load data and reuse the same train/val split

In [76]:
raw_df = pd.read_csv('/Users/nadiiababanska/Desktop/claude_coowork/ML_final_project/quora_with_features.csv', index_col=0)

X = raw_df.drop(columns=['is_duplicate'])
y = raw_df['is_duplicate']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

## Model 3 errors

### Sample val for local (CPU) inference

In [77]:
ERROR_ANALYSIS_SAMPLE_SIZE = 3000  # adjust up/down depending on how long CPU encoding takes

val_sample_df, _ = train_test_split(
    X_val.assign(is_duplicate=y_val),
    train_size=ERROR_ANALYSIS_SAMPLE_SIZE,
    stratify=y_val,
    random_state=42,
)

### Load the saved model and encode

In [78]:
st_model = SentenceTransformer('/Users/nadiiababanska/Desktop/claude_coowork/ML_final_project/models/sentence_transformer_duplicate_model')

q1_emb = st_model.encode(val_sample_df['question1'].tolist(), show_progress_bar=True)
q2_emb = st_model.encode(val_sample_df['question2'].tolist(), show_progress_bar=True)

val_sample_df = val_sample_df.copy()
val_sample_df['similarity'] = 1 - paired_cosine_distances(q1_emb, q2_emb)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/94 [00:00<?, ?it/s]

Batches:   0%|          | 0/94 [00:00<?, ?it/s]

### Pull false positives and false negatives

Threshold 0.60 -- the same one chosen for Model 3 in `04_sentence_transformers.ipynb` (from `reports/experiment_table.csv`).

In [79]:
MODEL_3_THRESHOLD = 0.60

false_positives = find_mismatched_cases(
    val_sample_df, score_col='similarity', threshold=MODEL_3_THRESHOLD,
    direction='above', target_label=0,
)
false_negatives = find_mismatched_cases(
    val_sample_df, score_col='similarity', threshold=MODEL_3_THRESHOLD,
    direction='below', target_label=1,
)

print(f'{len(false_positives)} false positives, {len(false_negatives)} false negatives out of {len(val_sample_df)} sampled rows')

209 false positives, 155 false negatives out of 3000 sampled rows


### Look at examples

Read through a handful of each -- what do the false positives have in common (surface similarity without meaning-level equivalence)? What about the false negatives (real duplicates the model misses)?

In [80]:
false_positives[['question1', 'question2', 'similarity']].sort_values('similarity', ascending=False).head(15)

,question1,question2,similarity
id,,,
172365,"If you could afford the time and money, would you change your way of life? Why?","If you could afford the time and money, would you change your way of life?",0.997569
264893,What is the admission process for MIT?,What is the detailed admission procedure for MIT?,0.983543
2490,What are my chances of being admitted into the Cornell M.Eng. program in ECE?,What are my chances to be admitted by the Cornell M.Eng. program in ECE?,0.982423
336565,Clinton Family: What are the differences between Bill and Hillary as people?,Clinton Family: What are the differences between Bill and Hillary as politicians?,0.981867
229170,What should you ask your client when building a website?,What should I ask my client when building a website?,0.981395
129609,What is an SSL Certificate?,What is SSL certificate?,0.971838
64367,Has technology affected relationships?,How has technology affected relationships?,0.971724
320682,Where can I get best fire inspection services in Sydney?,Where can I get best fire life safety services in Sydney?,0.961920
140402,How do I quickly and efficiently learn a new language?,What are the most efficient ways to learn a new language?,0.961865


In [81]:
false_negatives[['question1', 'question2', 'similarity']].sort_values('similarity').head(15)

,question1,question2,similarity
id,,,
262369,How do I transfer contacts from iPhone to iPhone with ease?,How can I transfer contacts from iPhone 3GS to iPhone 4,0.013504
38005,I didn't have much appetite before Prozac and still don't have it one month in either. How does this make sense?,Why don't I still have much appetite after one month on Prozac? Is the drug working for me?,0.014492
20635,What is simulation theory in philosophy?,What is the simulation theory?,0.026425
320724,Social Etiquette: How long should I wait to text her?,How long should I wait to text him?,0.053628
356060,How is UNCC for MS in CS? How are the opportunities like TA & RA for CS graduates?,How good are TA opportunities for MS in Computer Science program in UNCC for spring admissions?,0.057847
199073,How can I install Mac OS in my HP Laptop?,Can I install os x yosemite on dell inspiron windows 10 laptop?,0.070020
310435,How effective are IP address 'blockers'/'hiders' for Mac?,How effective are IP address 'blockers'/'hiders'?,0.123770
152608,What is the best tool in E_Learning?,What are the best elearning tools?,0.125231
119678,Where can I find a surf (Speeded Up Robust Features) MATLAB Code for Keypoint detector and keypoint descriptor?,Where can I find Code of SURF keypoint detector & keypoint descriptor in MATLAB?,0.183097


### Manual label review

Quora Question Pairs has known label noise from its crowdsourced annotation -- some "errors" above may actually be mislabeled pairs, not real model mistakes. Spot-check a sample of the false positives/negatives by hand: for each, judge whether the *true label* looks right to you, independent of what the model predicted.

This gives a rough estimate of how much of the apparent error rate is label noise vs. genuine model limitations -- useful context for the README conclusions (a model's raw log loss/F1 can understate its real quality if it's being penalized for correctly judging mislabeled pairs).

In [82]:
REVIEW_SAMPLE_SIZE = 20  # per category (FP/FN), adjust as needed

review_df = pd.concat([
    false_positives[['question1', 'question2', 'similarity']].assign(error_type='false_positive').sample(
        min(REVIEW_SAMPLE_SIZE, len(false_positives)), random_state=42
    ),
    false_negatives[['question1', 'question2', 'similarity']].assign(error_type='false_negative').sample(
        min(REVIEW_SAMPLE_SIZE, len(false_negatives)), random_state=42
    ),
])
review_df

,question1,question2,similarity,error_type
id,,,,
202120,How do you get rid of bees in the house?,How can I get rid of a bee nest in the house?,0.865024,false_positive
402083,What does SSL mean?,What is HTTPS/SSL?,0.698004,false_positive
341397,Can I give my dog Benadryl to help him calm down?,Can I give Benadryl to help my baby sleep?,0.623552,false_positive
267889,Is the finisher Dhoni finished?,Is Dhoni really a match finisher?,0.737115,false_positive
47040,"Is diet green tea good for you? If so, why?",Is green tea good for health?,0.827521,false_positive
65543,How hard was it for you to learn how to play the guitar?,Is it hard to learn how to play the piano?,0.613535,false_positive
347501,What's the best way to break up with someone you are not in love with?,How can I break up with someone I love but don't love me?,0.609846,false_positive
315037,What does it take to become a TV show host?,How do one become a host for a TV travel show?,0.659957,false_positive
114997,Which is the best Toastmasters club in Pune?,Where can I find toastmasters clubs in pune?,0.958228,false_positive


Fill in your judgment for each row below, keyed by its `id` (the DataFrame index shown in the table above) -- `True` if the true label looks correct to you (so it's a genuine model mistake), `False` if the label looks wrong (the model's prediction was actually more reasonable than the label).

In [83]:
label_review = {
    202120:False,
    402083:True,
    341397:True,
    267889:True,
    47040:False,
    65543:True,
    347501:True,
    315037:True,
    114997:True,
    66355:True,
    179994:False,
    239919:True,
    8696:True,
    398303:False,
    47962:True,
    139937:True,
    157540:True,
    246907:True,
    344586:True,
    264607:False,
    74320:False,
    283629:True,
    133588:True,
    351757:True,
    206578:True,
    182466:True,
    7914:False,
    1515:False,
    41865:True,
    183758:True,
    197719:False,
    132789:True,
    325600:True,
    151989:True,
    267502:True,
    52113:True,
    297320:True,
    321599:True,
    114874:True,
    283463:True

}

review_df['label_looks_correct'] = review_df.index.map(label_review)

In [84]:
reviewed = review_df.dropna(subset=['label_looks_correct'])
pct_label_noise = (1 - reviewed['label_looks_correct'].mean()) * 100
print(f'{len(reviewed)} rows reviewed -- estimated {pct_label_noise:.0f}% look like label noise rather than genuine model errors')

40 rows reviewed -- estimated 22% look like label noise rather than genuine model errors


#### Observations
Model has mistakes on such types of pairs:
- where the question is about the same topic but there is some context that can influence
- where synonims are used
- with misprints
- when the question is on the same topic but in one case it's direct question in other it's opinion question
- when question words change the expected result of question
- when there are abbreviations

## Model 4 (LLM few-shot) errors

Reuses the cached responses from `05_llm_fewshot.ipynb` (`reports/llm_fewshot_raw_responses.jsonl`) -- no new API calls needed, this data was already collected during that notebook's evaluation run.

In [85]:
llm_results_df = pd.read_json(
    '/Users/nadiiababanska/Desktop/claude_coowork/ML_final_project/reports/llm_fewshot_raw_responses.jsonl',
    lines=True,
)

# bring question1/question2 back in by joining on id (the original DataFrame index saved in row.Index during scoring)
llm_results_df = llm_results_df.merge(
    raw_df[['question1', 'question2']], left_on='id', right_index=True, how='left'
)

In [86]:
MODEL_4_THRESHOLD = 0.5

llm_false_positives = find_mismatched_cases(
    llm_results_df, score_col='score', threshold=MODEL_4_THRESHOLD,
    direction='above', target_label=0,
)
llm_false_negatives = find_mismatched_cases(
    llm_results_df, score_col='score', threshold=MODEL_4_THRESHOLD,
    direction='below', target_label=1,
)

print(f'{len(llm_false_positives)} false positives, {len(llm_false_negatives)} false negatives out of {len(llm_results_df)} evaluated rows')

38 false positives, 27 false negatives out of 505 evaluated rows


In [87]:
llm_false_positives[['question1', 'question2', 'score']].sort_values('score', ascending=False)

,question1,question2,score
282,Where can I find investors for my start up idea?,How do I find investors for my startup?,0.95
215,At what altitude do you see the curvature of the Earth?,At what altitude (in clear conditions) can the curvature of the earth be discerned by the naked eye?,0.92
372,Why do people say bless you when you sneeze?,"What is the origin of saying ""bless you"" when someone sneezes?",0.92
18,Which is the best Mobile app development company in USA?,Which is the best mobile app development company in USA 2016?,0.85
445,How can you tell if a woman is a gold digger?,How do I weed out gold diggers?,0.85
433,Is there any harm in masturbating?,"What are the side effects (positive and negative), if any, of masturbation?",0.85
368,Should I update to Marshmallow from lollipop?,How do I update lollipop to marshmallow?,0.85
365,Which start up would be best to start with?,Which is best start up?,0.85
341,In India why the manufacturers don’t use diesel engine for two wheeler?,What are the major reasons that two-wheelers and motorbikes are not designed on diesel engines?,0.85
264,When will India-Pakistan war end?,How will the India-Pakistan battle end?,0.85


In [88]:
llm_false_negatives[['question1', 'question2', 'score']].sort_values('score')

,question1,question2,score
7,How would a military coup in the US be foiled?,How likely is a military coup in the United States?,0.15
11,Silicon Valley Bank (SVB): What exactly is unique to Silicon Valley Bank?,Silicon Valley Bank (SVB): What banks are similar to Silicon Valley Bank?,0.15
478,Is time travel to 2010 possible?,When will time travelling (or at least time shifted vision) finally be possible?,0.15
459,Will there be a nuclear war between India and Pakistan?,What will be the effect of possible war between India and Pakistan on Indian Stock market?,0.15
125,Can the existence of God be either proved or negated?,How do I prove the existence of God to an atheist?,0.15
259,The question was marked as needing improvement. Just now?,What should I do if my question is being marked instantly as needing improvement but I don't know why?,0.15
488,What's the best thing that's ever happened to you?,What is the most interesting thing that happened to you?,0.25
416,Is bulletproof coffee campaign legit?,Has bulletproof coffee worked for those who have drunk it?,0.25
401,How does sex feels like?,How was Sex for you?,0.25
494,Does the FBI have Hillary Clinton under surveillance?,Why has the FBI reopened the investigation into Hillary Clinton's emails?,0.25


### Manual label review (Model 4)

Same idea as the Model 3 review above -- spot-check a sample of Model 4's false positives/negatives and judge whether the *true label* looks right, independent of what the model predicted. Model 4 was only evaluated on 500 rows, so there may be fewer errors to sample from than the `REVIEW_SAMPLE_SIZE` requested -- `min(...)` below handles that.

In [89]:
LLM_REVIEW_SAMPLE_SIZE = 20  # per category (FP/FN)

llm_review_df = pd.concat([
    llm_false_positives[['question1', 'question2', 'score']].assign(error_type='false_positive').sample(
        min(LLM_REVIEW_SAMPLE_SIZE, len(llm_false_positives)), random_state=42
    ),
    llm_false_negatives[['question1', 'question2', 'score']].assign(error_type='false_negative').sample(
        min(LLM_REVIEW_SAMPLE_SIZE, len(llm_false_negatives)), random_state=42
    ),
])
llm_review_df

,question1,question2,score,error_type
433,Is there any harm in masturbating?,"What are the side effects (positive and negative), if any, of masturbation?",0.85,false_positive
453,"How do you distinguish fake, plastic rice imported from China from the real rice that doesn't have plastic in it?","Is there really ""fake rice made from plastic"" being exported by China?",0.72,false_positive
108,What can I use to substitute butter extract?,What can be substituted for butter?,0.72,false_positive
253,What is a homosexual?,What is homosexuality?,0.85,false_positive
388,How do I publish poetry on Quora?,Can I post my poetry on Quora?,0.75,false_positive
365,Which start up would be best to start with?,Which is best start up?,0.85,false_positive
128,Which is the best WooCommerce WordPress Theme in 2016?,What are the best woocommerce themes?,0.78,false_positive
367,Who are the best technical recruiters in London?,Who are the best engineering recruiters in London?,0.72,false_positive
341,In India why the manufacturers don’t use diesel engine for two wheeler?,What are the major reasons that two-wheelers and motorbikes are not designed on diesel engines?,0.85,false_positive
264,When will India-Pakistan war end?,How will the India-Pakistan battle end?,0.85,false_positive


Fill in your judgment for each row below, keyed by its `id` (the DataFrame index shown in the table above) -- `True` if the true label looks correct to you (genuine model mistake), `False` if the label looks wrong (the model's prediction was actually more reasonable than the label).

In [90]:
llm_label_review = {
    433:False,
    453:True,
    108:True,
    253:True,
    388:True,
    365:True,
    128:True,
    367:False,
    341:False,
    264:True,
    282:False,
    175:True,
    267:False,
    248:False,
    300:True,
    183:False,
    428:True,
    18:True,
    355:True,
    112:False,
    169:True,
    349:True,
    220:True,
    459:False,
    7:False,
    285:False,
    401:True,
    416:False,
    299:False,
    488:False,
    11:False,
    89:True,
    111:False,
    20:True,
    375:True,
    467:False,
    49:False,
    494:True,
    478:True,
    419:False

}

llm_review_df['label_looks_correct'] = llm_review_df.index.map(llm_label_review)

In [91]:
llm_reviewed = llm_review_df.dropna(subset=['label_looks_correct'])
llm_pct_label_noise = (1 - llm_reviewed['label_looks_correct'].mean()) * 100
print(f'{len(llm_reviewed)} rows reviewed -- estimated {llm_pct_label_noise:.0f}% look like label noise rather than genuine model errors')

40 rows reviewed -- estimated 48% look like label noise rather than genuine model errors


#### Observations
LLM model makes in general the same types of mistakes and additionally:
- some context (even 1 word) can change the meaning of all sentence but model can't detect it
- when there's number in sentence and it differs, model would say it's the duplicate, however - the number can change the expected result from the question

### Cross-check: how does Model 3 score Model 4's number-related failures?

Model 4's false positives/negatives that involve a numeric detail (e.g. "top 10" vs "top 5") -- checking how Model 3 scores these *same* pairs. These specific pairs are very unlikely to already be in `val_sample_df` (a different, smaller stratified subsample), so instead of trying to look them up by `id`, re-encode them directly by question text with the already-loaded `st_model` -- text is the reliable join key here, not `id`.

In [92]:
llm_errors = pd.concat([llm_false_positives, llm_false_negatives])
has_number = llm_errors['question1'].str.contains(r'\d', regex=True) | llm_errors['question2'].str.contains(r'\d', regex=True)
llm_number_errors = llm_errors[has_number]

print(f'{len(llm_number_errors)} of {len(llm_errors)} Model 4 errors involve a number in question1 or question2')
llm_number_errors[['question1', 'question2', 'score', 'is_duplicate']]

7 of 65 Model 4 errors involve a number in question1 or question2


,question1,question2,score,is_duplicate
18,Which is the best Mobile app development company in USA?,Which is the best mobile app development company in USA 2016?,0.85,0
128,Which is the best WooCommerce WordPress Theme in 2016?,What are the best woocommerce themes?,0.78,0
267,How do I make money as a 15 year old?,How do I make money as a 14 year old?,0.75,0
319,Why does February have 28 days? Why 29 in leap years?,Why do we have a leap year?,0.65,0
169,Career Advice: What are the success tricks for preparing Gate in 3 months?,Can I crack gate CS 2017 in 3 months?,0.35,1
299,What are the best bikes on 2016?,What are the best bike inventions of 2016?,0.35,1
478,Is time travel to 2010 possible?,When will time travelling (or at least time shifted vision) finally be possible?,0.15,1


In [93]:
q1_emb_check = st_model.encode(llm_number_errors['question1'].tolist())
q2_emb_check = st_model.encode(llm_number_errors['question2'].tolist())

cross_check_df = llm_number_errors[['question1', 'question2', 'is_duplicate']].copy()
cross_check_df['model_4_score'] = llm_number_errors['score']
cross_check_df['model_3_similarity'] = 1 - paired_cosine_distances(q1_emb_check, q2_emb_check)
cross_check_df['model_3_would_predict'] = (cross_check_df['model_3_similarity'] > MODEL_3_THRESHOLD).astype(int)
cross_check_df

,question1,question2,is_duplicate,model_4_score,model_3_similarity,model_3_would_predict
18,Which is the best Mobile app development company in USA?,Which is the best mobile app development company in USA 2016?,0,0.85,0.630848,1
128,Which is the best WooCommerce WordPress Theme in 2016?,What are the best woocommerce themes?,0,0.78,0.119573,0
267,How do I make money as a 15 year old?,How do I make money as a 14 year old?,0,0.75,0.392644,0
319,Why does February have 28 days? Why 29 in leap years?,Why do we have a leap year?,0,0.65,0.275321,0
169,Career Advice: What are the success tricks for preparing Gate in 3 months?,Can I crack gate CS 2017 in 3 months?,1,0.35,0.560943,0
299,What are the best bikes on 2016?,What are the best bike inventions of 2016?,1,0.35,0.847011,1
478,Is time travel to 2010 possible?,When will time travelling (or at least time shifted vision) finally be possible?,1,0.15,0.813529,1


numer of questions - 7
label mistakes - 2
model 3 mistakes - 4
model 4 mistakes - 5
Model 3 and model 4 have the same weakness in processing question where some numbers are.

## Summary

**Shared failure pattern.** Both models key off topical/lexical overlap more than the underlying question actually calls for, and both under-weight small details that change what's being asked even when most of the wording overlaps. Concretely, both models' errors cluster around:
- Same topic, different sub-question -- one is a direct/factual question, the other an opinion question, or a qualifying detail changes what's really being asked
- Genuine synonym/paraphrase duplicates that share little surface wording
- Typos and abbreviations disrupting what should be an easy match
- Question words that flip the expected answer (e.g. "how" vs "why") despite otherwise near-identical phrasing

**Numeric details: no clear winner between the two models.** A direct cross-check -- re-scoring Model 4's number-related errors with Model 3 on the exact same pairs showed that both models handle it inconsistently.

**Label noise is substantial, and asymmetric between the two models.** Manual review of a 40-example sample estimated ~22% of Model 3's "errors" and ~48% of Model 4's "errors" are actually mislabeled pairs, not real model mistakes. Two implications:
    - Both models' raw log loss/F1 likely understate real quality somewhat -- part of what's being penalized as "wrong" is a reasonable judgment against a bad label.
    - The much higher noise share for Model 4 is notable but should be read cautiously: it's based on a smaller error pool (Model 4 was only scored on 500 rows total) and could reflect real strength (Claude's few-shot judgment more often lands on the *semantically* correct answer even when the dataset label disagrees) or could just be sampling noise from the small review set. Not a firm conclusion on its own -- worth a mention as a hypothesis in the README, not a claim.

**Takeaway for README Conclusions:** the biggest lever left for either model isn't more data or a bigger model -- it's better handling of small-but-meaningful context (opinion vs. fact framing, numeric details, question-word swaps). This is a harder problem than surface similarity, and is exactly where semantic/LLM-based approaches (Models 3-4) already outperform the lexical-overlap-based ones (Models 0-2), even if they haven't solved it completely.